### This notebook contains my final model attempt of getting the highest accuracy using retreival augmented fact checking (scraping the internet for facts)

##### I tried many different methods with BERT to try and train it and use it to classify misinformation however I hit a wall and could not improve the model any further. I know am going to attempt to make a retreival augmented model by utilizing wikipedia and DeBERTa-v3-base

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoTokenizer
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
device

c:\Users\13313\misinformationClassification\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cuda'

In [2]:
# df = pd.read_csv("combinedClaimDataset.csv")
# df.head()

In [3]:
# # Normalize source column
# df["source"] = df["source"].str.lower()

# # Separate FEVER and non-FEVER
# fever_df = df[df["source"] == "fever"]
# other_df = df[df["source"] != "fever"]

# print("Original FEVER size:", len(fever_df))
# print("Original OTHER size:", len(other_df))

# TARGET_FEVER_SIZE = 20000

# fever_down = fever_df.sample(
#     n=min(TARGET_FEVER_SIZE, len(fever_df)),
#     random_state=42
# )

# df_balanced = pd.concat([fever_down, other_df]).sample(frac=1, random_state=42).reset_index(drop=True)

In [4]:
# if df_balanced["label"].dtype == "object":
#     label2id = {lbl: i for i, lbl in enumerate(sorted(df_balanced["label"].unique()))}
#     id2label = {i: lbl for lbl, i in label2id.items()}
#     df_balanced["label"] = df_balanced["label"].map(label2id)
# else:
#     num_labels = df_balanced["label"].nunique()
#     id2label = {i: str(i) for i in range(num_labels)}
#     label2id = {v: k for k, v in id2label.items()}

In [5]:
# train_df, val_df = train_test_split(
#     df_balanced,
#     test_size=0.4,
#     random_state=42,
#     stratify=df_balanced["label"]
# )
# train_df = train_df.reset_index(drop=True)
# val_df = val_df.reset_index(drop=True)

In [6]:
# Sentence transformer for evidence retrieval
# embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# corpus = df["claim"].tolist()
# corpus_embeddings = embedder.encode(corpus, convert_to_numpy=True, show_progress_bar=True)

In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity
# def retrieve_semantic_evidence(claim, k=3):
#     claim_emb = embedder.encode([claim], convert_to_numpy=True)
#     sims = cosine_similarity(claim_emb, corpus_embeddings)[0]
#     top_idx = sims.argsort()[-k:][::-1]
#     evidence = " ".join([corpus[i] for i in top_idx])
#     return evidence

In [8]:
# Processing datasets and getting evidence
# from tqdm.auto import tqdm
# tqdm.pandas()
# train_df["evidence_text"] = train_df["claim"].progress_apply(retrieve_semantic_evidence)
# val_df["evidence_text"]   = val_df["claim"].progress_apply(retrieve_semantic_evidence)
# train_df["evidence_text"] = train_df["evidence_text"].fillna("").astype(str)
# val_df["evidence_text"]   = val_df["evidence_text"].fillna("").astype(str)

In [9]:
# Save the processed datasets so we never recompute evidence again
# train_df.to_csv("train_with_evidence.csv", index=False)
# val_df.to_csv("val_with_evidence.csv", index=False)

In [10]:
# use the already retrieved datasets so that we dont have to run it every time for an hour.
train_df = pd.read_csv("train_with_evidence.csv")
val_df   = pd.read_csv("val_with_evidence.csv")
print(train_df.columns)
train_df.head()

Index(['claim', 'label', 'source', 'evidence_text'], dtype='object')


,claim,label,source,evidence_text
0,The Democratic health care law added 12 years ...,0,liar,The Democratic health care law added 12 years ...
1,If Sen. Hillary Clinton could enact all of her...,2,liar,If Sen. Hillary Clinton could enact all of her...
2,Inferno (2016 film) is not the third installme...,1,fever,Inferno (2016 film) is not the third installme...
3,Says Rick Perry wrote a newspaper item saying ...,0,liar,Says Rick Perry wrote a newspaper item saying ...
4,A Milli is an album rather than a song.,1,fever,A Milli is an album rather than a song. A Mill...


In [ ]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize(batch):
    return tokenizer(
        batch["claim"],
        batch["evidence_text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [12]:
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 13383/13383 [00:00<00:00, 16171.67 examples/s]


In [13]:
num_labels = train_df["label"].nunique()
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    use_safetensors=True,
    trust_remote_code=True
)
model.to("cuda")

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 7150.00it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.w

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [14]:
# Converting to ensure model is not input with NaN
model = model.float()                     # this converts all weights to FP32
model.half = lambda *args, **kwargs: model  # this prevents Trainer from forcing FP16
model.to("cuda")

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [15]:
batch_size = 8

training_args = TrainingArguments(
    output_dir="./deberta_rag_checkpoints",
    do_train=True,
    do_eval=True,
    learning_rate=1e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    eval_steps=1,
    fp16=False,
    save_total_limit=1,
    save_only_model=True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./final_deberta_model")

Step,Training Loss
100,1.078359
200,1.034527
300,0.950549
400,0.930787
500,0.859171
600,0.858135
700,0.879386
800,0.858813
900,0.838612
1000,0.834594


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]


In [17]:
import torch
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_preds = []
all_labels = []

for i in range(len(val_ds)):
    batch = val_ds[i]

    # move tensors to GPU and add batch dimension
    input_ids = batch["input_ids"].unsqueeze(0).to(device)
    attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
    labels = batch["labels"].item()

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1).item()

    all_preds.append(preds)
    all_labels.append(labels)

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="macro")

print("Accuracy:", acc)
print("F1 Macro:", f1)

Accuracy: 0.6674138832847643
F1 Macro: 0.6425672331460665


#### This marks the end of update 4. In the last two weeks I continued working with the DistilBERT model. I tried to utilize contrastive learning to improve performance but could not get above an F1 of .6 so I made the decision to try make a retrieval augmented model. I had also spent some time pushing my code to a repo so I could run the models on my GPU since it was estimating to take 19 hours on my school laptop.